IMPORTS

In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

LOAD DATASET

In [12]:
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv')

PREPARING THE DATASET

In [13]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [14]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [15]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

In [16]:
for col in df.select_dtypes(include='object').columns:
    print(col, df[col].unique())

lead_source <StringArray>
['organic_search', 'social_media', 'referral', 'paid_ads', nan, 'events']
Length: 6, dtype: str
industry <StringArray>
[   'technology',        'retail', 'manufacturing',     'education',
    'healthcare',             nan,       'finance']
Length: 7, dtype: str
employment_status <StringArray>
['employed', 'student', nan, 'unemployed', 'self_employed']
Length: 5, dtype: str
location <StringArray>
['europe', 'south_america', nan, 'north_america', 'africa', 'asia']
Length: 6, dtype: str


/tmp/ipykernel_11205/3255506791.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


In [17]:
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

QUESTION 1

In [18]:
df['industry'].mode()

0    technology
Name: industry, dtype: str

QUESTION 2

In [19]:
df[numerical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


SPLITTING THE DATA

In [21]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

QUESTION 3

In [22]:
from sklearn.metrics import mutual_info_score

def mi_score(series):
    return mutual_info_score(series, y_train)

mi = df_train[categorical].apply(mi_score)
mi.sort_values(ascending=False).round(2)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

QUESTION 4

In [23]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)


model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

y_pred = model.predict_proba(X_val)[:, 1]

conversion_decision = (y_pred >= 0.5)
print((y_val == conversion_decision).mean().round(2))

0.64


QUESTION 5

In [24]:

features = categorical + numerical

def train_and_score(feats):
    dv = DictVectorizer(sparse=False)

    X_train = dv.fit_transform(df_train[feats].to_dict(orient='records')) 
    X_val = dv.transform(df_val[feats].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    conversion_decision = (y_pred >= 0.5)
    accuracy = (y_val == conversion_decision).mean()
    
    return accuracy

baseline = train_and_score(features)

for f in features:
    reduced_feat = [x for x in features if x != f]
    acc = train_and_score(reduced_feat)
    print(f'{f:30s} acc={acc:.5f} diff={baseline - acc:.5f}')

lead_source                    acc=0.64200 diff=0.00300
industry                       acc=0.64500 diff=0.00000
employment_status              acc=0.64500 diff=0.00000
location                       acc=0.64500 diff=0.00000
annual_income                  acc=0.72400 diff=-0.07900
number_of_courses_viewed       acc=0.64300 diff=0.00200
interaction_count              acc=0.60100 diff=0.04400
lead_score                     acc=0.64400 diff=0.00100


QUESTION 6

In [25]:
def train_and_score(feats, c):
    dv = DictVectorizer(sparse=False)

    X_train = dv.fit_transform(df_train[feats].to_dict(orient='records')) 
    X_val = dv.transform(df_val[feats].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    conversion_decision = (y_pred >= 0.5)
    accuracy = (y_val == conversion_decision).mean()
    
    return accuracy

C_values = [0.000001, 0.00001, 0.0001, 0.001]
for c in C_values:
    accuracy = train_and_score(features, c)
    print(f"C={c:<10} accuracy={accuracy:.3f}")

C=1e-06      accuracy=0.598
C=1e-05      accuracy=0.598
C=0.0001     accuracy=0.613
C=0.001      accuracy=0.645
